In [ ]:
import pandas as pd
import numpy as np

from sklearn.cluster import KMeans

from sklearn.preprocessing import StandardScaler

from sklearn.decomposition import PCA

import matplotlib.pyplot as plt

import seaborn as sns

import joblib


# -----------------------------------
# LOAD DATASET
# -----------------------------------

print("Loading dataset...")


df = pd.read_csv(
    "../../datasets/final_master_dataset.csv"
)


# -----------------------------------
# FEATURES
# -----------------------------------

features = [
    "irradiance",
    "total_consumption",
    "roi",
    "solar_performance_score",
    "annual_savings",
    "urbanization_score"
]


X = df[features]


# -----------------------------------
# FEATURE SCALING
# -----------------------------------

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)


# -----------------------------------
# ELBOW METHOD
# -----------------------------------

print("\nCalculating inertia values...")


inertia = []

k_range = range(1, 11)

for k in k_range:

    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    kmeans.fit(X_scaled)

    inertia.append(kmeans.inertia_)


# -----------------------------------
# ELBOW PLOT
# -----------------------------------

plt.figure(figsize=(10,6))

plt.plot(
    k_range,
    inertia,
    marker="o"
)

plt.xlabel("Number of Clusters")

plt.ylabel("Inertia")

plt.title("Elbow Method")

plt.show()


# -----------------------------------
# FINAL MODEL
# -----------------------------------

print("\nTraining K-Means Model...")


kmeans = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)


clusters = kmeans.fit_predict(X_scaled)


# -----------------------------------
# ASSIGN CLUSTERS
# -----------------------------------

df["cluster"] = clusters


# -----------------------------------
# CLUSTER SUMMARY
# -----------------------------------

cluster_summary = (
    df.groupby("cluster")[features]
    .mean()
)

print("\nCluster Summary:")

print(cluster_summary)


# -----------------------------------
# PCA REDUCTION
# -----------------------------------

pca = PCA(n_components=2)

X_pca = pca.fit_transform(X_scaled)


# -----------------------------------
# CLUSTER VISUALIZATION
# -----------------------------------

plt.figure(figsize=(12,8))

scatter = plt.scatter(
    X_pca[:,0],
    X_pca[:,1],
    c=clusters
)

plt.xlabel("PCA Component 1")

plt.ylabel("PCA Component 2")

plt.title("Regional Solar Clusters")

plt.legend(
    *scatter.legend_elements(),
    title="Clusters"
)

plt.show()


# -----------------------------------
# SAVE DATASET
# -----------------------------------

df.to_csv(
    "../../datasets/clustered_dataset.csv",
    index=False
)


# -----------------------------------
# SAVE MODEL
# -----------------------------------

joblib.dump(
    kmeans,
    "../../trained_models/kmeans_clustering_model.pkl"
)

print("\nClustering model saved successfully.")

: 